<a href="https://colab.research.google.com/github/SantiagoDiaz008/ALGORITMO-Y-PROGRAMACION/blob/main/ABP_ALGORITMO_Y_PROGRAMACION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from fpdf import FPDF
from google.colab import files
import matplotlib.pyplot as plt

LINEA = "=" * 52
SEPARADOR = "-" * 52
PORCENTAJE_SOBRECONSUMO = 0.15


def leer_texto(mensaje):
    texto = input(mensaje).strip()
    while texto == "":
        print("  [!] Este campo no puede estar vacío.")
        texto = input(mensaje).strip()
    return texto


def leer_real(mensaje, permitir_cero=False):
    while True:
        try:
            valor = float(input(mensaje).replace(",", "."))
        except ValueError:
            print("  [!] Debe ingresar un número válido.")
            continue

        if valor < 0 or (valor == 0 and not permitir_cero):
            print("  [!] El valor debe ser mayor que cero.")
        else:
            return valor


def capturar_datos():
    print("\n" + LINEA)
    print(" REGISTRO DE UNA MÁQUINA")
    print(LINEA)

    maquina = {
        "id_equipo": leer_texto("Ingrese el ID o código de la máquina: "),
        "tipo_maquina": leer_texto(
            "Ingrese el tipo de máquina (Ej: Motor, Compresor, Excavadora): "
        ),
        "horas_trabajo": leer_real("Ingrese las horas de trabajo acumuladas (hrs): "),
        "consumo_energia": leer_real(
            "Ingrese el consumo total registrado (kWh o Galones): ", permitir_cero=True
        ),
        "costo_unidad_energia": leer_real(
            "Ingrese el costo por unidad de energía ($/kWh o $/Galón): ",
            permitir_cero=True,
        ),
        "limite_horas_mantenimiento": leer_real(
            "Ingrese el límite máximo de horas para mantenimiento preventivo: "
        ),
        "consumo_estandar_hora": leer_real(
            "Ingrese el consumo estándar esperado por hora (según fabricante): ",
            permitir_cero=True,
        ),
    }
    return maquina


def evaluar_maquina(m):
    m["costo_total"] = m["consumo_energia"] * m["costo_unidad_energia"]
    m["costo_por_hora"] = m["costo_total"] / m["horas_trabajo"]
    m["consumo_promedio_hora"] = m["consumo_energia"] / m["horas_trabajo"]
    m["sobreconsumo"] = m["consumo_promedio_hora"] - m["consumo_estandar_hora"]

    if m["sobreconsumo"] > 0:
        m["estado_energetico"] = "ALERTA DE INEFICIENCIA"
    else:
        m["estado_energetico"] = "ÓPTIMO"

    limite_sobreconsumo = m["consumo_estandar_hora"] * PORCENTAJE_SOBRECONSUMO
    exceso_horas = m["horas_trabajo"] >= m["limite_horas_mantenimiento"]
    exceso_consumo = m["sobreconsumo"] > limite_sobreconsumo

    if exceso_horas and exceso_consumo:
        m["estado_mantenimiento"] = "REQUIERE MANTENIMIENTO URGENTE"
        m["motivo"] = "Exceso de horas acumuladas y sobreconsumo crítico."
    elif exceso_horas:
        m["estado_mantenimiento"] = "REQUIERE MANTENIMIENTO URGENTE"
        m["motivo"] = "Exceso de horas acumuladas."
    elif exceso_consumo:
        m["estado_mantenimiento"] = "REQUIERE MANTENIMIENTO URGENTE"
        m["motivo"] = "Sobreconsumo crítico (más del 15 % sobre el estándar)."
    else:
        m["estado_mantenimiento"] = "EN REGLA"
        m["motivo"] = "El equipo no requiere mantenimiento por el momento."

    if exceso_horas:
        m["horas_faltantes"] = 0.0
    else:
        m["horas_faltantes"] = m["limite_horas_mantenimiento"] - m["horas_trabajo"]

    return m


def mostrar_informe(m):
    color_estado = "#EF4444" if m['estado_mantenimiento'] != "EN REGLA" else "#10B981"
    color_energetico = "#F59E0B" if m['sobreconsumo'] > 0 else "#10B981"

    detalle_sobreconsumo = (
        f"El equipo presenta un sobreconsumo de {m['sobreconsumo']:,.4f} por hora."
        if m["sobreconsumo"] > 0
        else "El consumo está dentro del rango esperado."
    )

    if m["horas_faltantes"] == 0:
        proximo_mant = "El mantenimiento está pendiente. Se recomienda realizarlo inmediatamente."
    else:
        proximo_mant = f"Faltan aproximadamente {m['horas_faltantes']:,.2f} horas de trabajo. Se recomienda realizar el mantenimiento al llegar a {m['limite_horas_mantenimiento']:,.2f} horas."

    html_informe = f"""
    <div style="background-color: #1E293B; padding: 20px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; color: #F8FAFC; border: 1px solid #475569; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
        <h3 style="color: #38BDF8; margin-top: 0; border-bottom: 2px solid #334155; padding-bottom: 10px;">📑 INFORME DE RESULTADOS</h3>
        <p style="font-size: 15px; font-weight: bold; margin-bottom: 15px;">Máquina evaluada: <span style="color: #38BDF8;">{m['id_equipo']}</span> ({m['tipo_maquina']}) — <span style="color: #94A3B8;">{m['horas_trabajo']:,.2f} hrs</span></p>

        <div style="background-color: #0F172A; padding: 12px; border-radius: 8px; margin-bottom: 10px;">
            <strong style="color: #38BDF8;">💰 ANÁLISIS ECONÓMICO:</strong>
            <ul style="margin: 5px 0 0 20px; color: #CBD5E1;">
                <li>Costo total de operación: <b>${m['costo_total']:,.2f}</b></li>
                <li>Costo operativo por hora: <b>${m['costo_por_hora']:,.2f}</b></li>
            </ul>
        </div>

        <div style="background-color: #0F172A; padding: 12px; border-radius: 8px; margin-bottom: 10px;">
            <strong style="color: #38BDF8;">⚡ DESEMPEÑO ENERGÉTICO:</strong>
            <ul style="margin: 5px 0 0 20px; color: #CBD5E1;">
                <li>Consumo promedio por hora: <b>{m['consumo_promedio_hora']:,.4f}</b></li>
                <li>Consumo estándar esperado: <b>{m['consumo_estandar_hora']:,.4f}</b></li>
                <li>Estado energético: <span style="color: {color_energetico}; font-weight: bold;">[{m['estado_energetico']}]</span> ({detalle_sobreconsumo})</li>
            </ul>
        </div>

        <div style="background-color: #0F172A; padding: 12px; border-radius: 8px; margin-bottom: 10px;">
            <strong style="color: #38BDF8;">🛠️ DIAGNÓSTICO DE MANTENIMIENTO:</strong>
            <ul style="margin: 5px 0 0 20px; color: #CBD5E1;">
                <li>Estado: <span style="color: {color_estado}; font-weight: bold;">[{m['estado_mantenimiento']}]</span></li>
                <li>Motivo: {m['motivo']}</li>
            </ul>
        </div>

        <div style="background-color: #0F172A; padding: 12px; border-radius: 8px;">
            <strong style="color: #38BDF8;">⏰ PRÓXIMO MANTENIMIENTO:</strong>
            <p style="margin: 5px 0 0 20px; color: #CBD5E1;">{proximo_mant}</p>
        </div>
    </div>
    """
    display(HTML(html_informe))


def mostrar_resumen(flota):
    if len(flota) == 0:
        display(HTML("""
        <div style="background-color: #1E293B; padding: 15px; border-radius: 8px; color: #F8FAFC; border-left: 4px solid #F59E0B; font-family: 'Segoe UI', sans-serif;">
            <strong>[!] Aún no hay máquinas registradas en la flota.</strong>
        </div>
        """))
        return

    html_tabla = """
    <div style="background-color: #1E293B; padding: 20px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; color: #F8FAFC; border: 1px solid #475569;">
        <h3 style="color: #38BDF8; margin-top: 0; border-bottom: 2px solid #334155; padding-bottom: 10px;">📊 RESUMEN DE MÁQUINAS REGISTRADAS</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 14px;">
            <thead>
                <tr style="background-color: #334155; color: #38BDF8; text-align: left;">
                    <th style="padding: 10px; border-top-left-radius: 6px;">N°</th>
                    <th style="padding: 10px;">ID Equipo</th>
                    <th style="padding: 10px;">Tipo</th>
                    <th style="padding: 10px;">Costo / Hora</th>
                    <th style="padding: 10px; border-top-right-radius: 6px;">Estado Mantenimiento</th>
                </tr>
            </thead>
            <tbody>
    """

    for i, m in enumerate(flota, start=1):
        color_estado = "#EF4444" if m['estado_mantenimiento'] != "EN REGLA" else "#10B981"
        bg_fila = "#0F172A" if i % 2 == 0 else "#1E293B"
        html_tabla += f"""
                <tr style="background-color: {bg_fila}; border-bottom: 1px solid #334155;">
                    <td style="padding: 10px;">{i}</td>
                    <td style="padding: 10px; font-weight: bold; color: #38BDF8;">{m['id_equipo']}</td>
                    <td style="padding: 10px;">{m['tipo_maquina']}</td>
                    <td style="padding: 10px;">${m['costo_por_hora']:,.2f}</td>
                    <td style="padding: 10px; color: {color_estado}; font-weight: bold;">{m['estado_mantenimiento']}</td>
                </tr>
        """

    html_tabla += """
            </tbody>
        </table>
    </div>
    """
    display(HTML(html_tabla))


def mostrar_comparacion(flota):
    if len(flota) < 2:
        display(HTML("""
        <div style="background-color: #1E293B; padding: 15px; border-radius: 8px; color: #F8FAFC; border-left: 4px solid #F59E0B; font-family: 'Segoe UI', sans-serif;">
            <strong>[!] Registre al menos 2 máquinas para poder compararlas.</strong>
        </div>
        """))
        return

    mas_cara = flota[0]
    mas_barata = flota[0]
    urgentes = 0
    total_costos = 0.0

    for m in flota:
        total_costos += m["costo_total"]
        if m["costo_por_hora"] > mas_cara["costo_por_hora"]:
            mas_cara = m
        if m["costo_por_hora"] < mas_barata["costo_por_hora"]:
            mas_barata = m
        if m["estado_mantenimiento"] != "EN REGLA":
            urgentes += 1

    html_comparacion = f"""
    <div style="background-color: #1E293B; padding: 20px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; color: #F8FAFC; border: 1px solid #475569;">
        <h3 style="color: #38BDF8; margin-top: 0; border-bottom: 2px solid #334155; padding-bottom: 10px;">⚖️ COMPARACIÓN ENTRE MÁQUINAS</h3>

        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-top: 15px;">
            <div style="background-color: #0F172A; padding: 15px; border-radius: 8px;">
                <span style="color: #94A3B8; font-size: 12px; font-weight: bold;">MÁQUINAS EVALUADAS</span>
                <p style="font-size: 20px; font-weight: bold; margin: 5px 0 0 0; color: #38BDF8;">{len(flota)} Equipos</p>
            </div>
            <div style="background-color: #0F172A; padding: 15px; border-radius: 8px;">
                <span style="color: #94A3B8; font-size: 12px; font-weight: bold;">COSTO TOTAL ACUMULADO</span>
                <p style="font-size: 20px; font-weight: bold; margin: 5px 0 0 0; color: #10B981;">${total_costos:,.2f}</p>
            </div>
        </div>

        <div style="background-color: #0F172A; padding: 15px; border-radius: 8px; margin-top: 15px;">
            <p style="margin: 5px 0;"><strong>🔴 Más costosa por hora:</strong> <span style="color: #EF4444;">{mas_cara['id_equipo']} ({mas_cara['tipo_maquina']}) - ${mas_cara['costo_por_hora']:,.2f}/h</span></p>
            <p style="margin: 10px 0 5px 0;"><strong>🟢 Más económica por hora:</strong> <span style="color: #10B981;">{mas_barata['id_equipo']} ({mas_barata['tipo_maquina']}) - ${mas_barata['costo_por_hora']:,.2f}/h</span></p>
            <p style="margin: 10px 0 5px 0;"><strong>⚠️ Requieren mantenimiento:</strong> <span style="color: #F59E0B;">{urgentes} de {len(flota)} máquinas</span></p>
        </div>
    </div>
    """
    display(HTML(html_comparacion))


def mostrar_grafica(flota):
    if len(flota) == 0:
        display(HTML("""
        <div style="background-color: #1E293B; padding: 15px; border-radius: 8px; color: #F8FAFC; border-left: 4px solid #F59E0B; font-family: 'Segoe UI', sans-serif;">
            <strong>[!] Registre al menos 1 máquina para generar la gráfica estadística.</strong>
        </div>
        """))
        return

    ids = [m['id_equipo'] for m in flota]
    costos = [m['costo_por_hora'] for m in flota]
    colores = ['#EF4444' if m['estado_mantenimiento'] != "EN REGLA" else '#10B981' for m in flota]

    plt.figure(figsize=(9, 4.5))
    plt.bar(ids, costos, color=colores, width=0.5, edgecolor='#334155', linewidth=1.5)
    plt.title("Costo Operativo por Hora ($/h) - Flota de Máquinas", fontsize=12, fontweight='bold', color='#0F172A', pad=15)
    plt.xlabel("ID del Equipo", fontsize=10, fontweight='bold', color='#334155')
    plt.ylabel("Costo por Hora ($)", fontsize=10, fontweight='bold', color='#334155')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    display(HTML("""
    <div style="background-color: #1E293B; padding: 12px; border-radius: 8px; color: #10B981; border-left: 4px solid #10B981; font-family: 'Segoe UI', sans-serif;">
        <strong>✅ Gráfica estadística generada con éxito (Rojo = Urgente, Verde = En regla).</strong>
    </div>
    """))


def exportar_pdf(flota):
    if len(flota) == 0:
        display(HTML("""
        <div style="background-color: #1E293B; padding: 15px; border-radius: 8px; color: #F8FAFC; border-left: 4px solid #F59E0B; font-family: 'Segoe UI', sans-serif;">
            <strong>[!] Registre al menos 1 máquina antes de descargar el PDF.</strong>
        </div>
        """))
        return

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_fill_color(30, 41, 59)
    pdf.rect(0, 0, 210, 32, 'F')
    pdf.set_text_color(56, 189, 248)
    pdf.set_font("Helvetica", "B", 15)
    pdf.set_y(10)
    pdf.cell(0, 8, "REPORTE DE CONTROL ENERGETICO Y MANTENIMIENTO", ln=True, align="C")
    pdf.set_text_color(203, 213, 225)
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 6, f"Total de equipos evaluados: {len(flota)}", ln=True, align="C")

    pdf.set_y(40)
    pdf.set_text_color(15, 23, 42)

    if len(flota) >= 2:
        mas_cara = max(flota, key=lambda x: x["costo_por_hora"])
        mas_barata = min(flota, key=lambda x: x["costo_por_hora"])
        urgentes = sum(1 for m in flota if m["estado_mantenimiento"] != "EN REGLA")
        total_costos = sum(m["costo_total"] for m in flota)

        pdf.set_font("Helvetica", "B", 12)
        pdf.set_text_color(2, 132, 199)
        pdf.cell(0, 8, "RESUMEN COMPARATIVO DE FLOTA", ln=True)
        pdf.set_font("Helvetica", "", 10)
        pdf.set_text_color(30, 41, 59)
        pdf.cell(0, 6, f"- Costo total acumulado de la flota: ${total_costos:,.2f}", ln=True)
        pdf.cell(0, 6, f"- Mas costosa por hora: {mas_cara['id_equipo']} ({mas_cara['tipo_maquina']}) - ${mas_cara['costo_por_hora']:,.2f}/h", ln=True)
        pdf.cell(0, 6, f"- Mas economica por hora: {mas_barata['id_equipo']} ({mas_barata['tipo_maquina']}) - ${mas_barata['costo_por_hora']:,.2f}/h", ln=True)
        pdf.cell(0, 6, f"- Equipos que requieren mantenimiento urgente: {urgentes} de {len(flota)}", ln=True)
        pdf.ln(6)

    for i, m in enumerate(flota, start=1):
        pdf.set_fill_color(241, 245, 249)
        pdf.set_font("Helvetica", "B", 11)
        pdf.set_text_color(15, 23, 42)
        pdf.cell(0, 8, f" {i}. EQUIPO: {m['id_equipo']} - {m['tipo_maquina']} ({m['horas_trabajo']:,.2f} hrs)", ln=True, fill=True)

        pdf.set_font("Helvetica", "", 10)
        pdf.cell(0, 6, f"   Costo Total: ${m['costo_total']:,.2f}   |   Costo/Hora: ${m['costo_por_hora']:,.2f}", ln=True)
        pdf.cell(0, 6, f"   Consumo Promedio/Hora: {m['consumo_promedio_hora']:,.4f}   |   Estandar: {m['consumo_estandar_hora']:,.4f}", ln=True)
        pdf.cell(0, 6, f"   Estado Energetico: {m['estado_energetico']}", ln=True)

        if m["estado_mantenimiento"] != "EN REGLA":
            pdf.set_text_color(220, 38, 38)
        else:
            pdf.set_text_color(5, 150, 105)
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(0, 6, f"   Diagnostico: {m['estado_mantenimiento']} - {m['motivo']}", ln=True)

        pdf.set_text_color(71, 85, 105)
        pdf.set_font("Helvetica", "I", 9)
        if m["horas_faltantes"] == 0:
            pdf.cell(0, 6, "   Recomendacion: Realizar mantenimiento preventivo inmediatamente.", ln=True)
        else:
            pdf.cell(0, 6, f"   Recomendacion: Faltan {m['horas_faltantes']:,.2f} hrs (limite: {m['limite_horas_mantenimiento']:,.2f} hrs).", ln=True)

        pdf.ln(4)

    nombre_archivo = "Informe_Maquinaria.pdf"
    pdf.output(nombre_archivo)
    files.download(nombre_archivo)

    display(HTML(f"""
    <div style="background-color: #1E293B; padding: 14px; border-radius: 8px; color: #10B981; border-left: 4px solid #10B981; font-family: 'Segoe UI', sans-serif;">
        <strong>✅ El archivo '{nombre_archivo}' se ha generado y descargado correctamente.</strong>
    </div>
    """))


def iniciar_interfaz():
    flota = []

    encabezado = widgets.HTML("""
    <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 20px; border-radius: 12px; border-left: 6px solid #38BDF8; font-family: 'Segoe UI', sans-serif; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
        <h2 style="color: #38BDF8; margin: 0 0 5px 0; font-size: 18px;">⚡ Control Energético y Mantenimiento Preventivo</h2>
        <p style="color: #94A3B8; margin: 0; font-size: 13px;">Sistema de diagnóstico y evaluación operativa para maquinaria industrial.</p>
    </div>
    """)

    estilo = {'description_width': '220px'}
    ancho = widgets.Layout(width='520px')

    in_id = widgets.Text(description="ID o código de la máquina:", style=estilo, layout=ancho)
    in_tipo = widgets.Text(description="Tipo de máquina:", style=estilo, layout=ancho)
    in_horas = widgets.Text(description="Horas de trabajo (hrs):", style=estilo, layout=ancho)
    in_consumo = widgets.Text(description="Consumo total (kWh/Gal):", style=estilo, layout=ancho)
    in_costo = widgets.Text(description="Costo por unidad ($):", style=estilo, layout=ancho)
    in_limite = widgets.Text(description="Límite máximo de horas:", style=estilo, layout=ancho)
    in_estandar = widgets.Text(description="Consumo estándar / hora:", style=estilo, layout=ancho)

    btn_registrar = widgets.Button(description="📥 Registrar", button_style='info', layout=widgets.Layout(width='125px', height='38px'))
    btn_resumen = widgets.Button(description="📊 Resumen", button_style='success', layout=widgets.Layout(width='115px', height='38px'))
    btn_comparar = widgets.Button(description="⚖️ Comparar", button_style='warning', layout=widgets.Layout(width='110px', height='38px'))
    btn_grafica = widgets.Button(description="📈 Gráfica", button_style='', layout=widgets.Layout(width='110px', height='38px', background_color='#8B5CF6'))
    btn_pdf = widgets.Button(description="📄 PDF", button_style='danger', layout=widgets.Layout(width='95px', height='38px'))

    salida = widgets.Output()

    def validar_real(texto, nombre, permitir_cero=False):
        try:
            valor = float(texto.replace(",", "."))
        except ValueError:
            raise ValueError(f"[!] '{nombre}': Debe ingresar un número válido.")
        if valor < 0 or (valor == 0 and not permitir_cero):
            raise ValueError(f"[!] '{nombre}': El valor debe ser mayor que cero.")
        return valor

    def on_registrar(b):
        with salida:
            clear_output()
            try:
                id_eq = in_id.value.strip()
                tipo_eq = in_tipo.value.strip()
                if id_eq == "" or tipo_eq == "":
                    display(HTML("""
                    <div style="background-color: #1E293B; padding: 12px; border-radius: 8px; color: #EF4444; border-left: 4px solid #EF4444; font-family: 'Segoe UI', sans-serif;">
                        <strong>[!] El ID y el tipo de máquina no pueden estar vacíos.</strong>
                    </div>
                    """))
                    return

                maquina = {
                    "id_equipo": id_eq,
                    "tipo_maquina": tipo_eq,
                    "horas_trabajo": validar_real(in_horas.value, "Horas de trabajo", False),
                    "consumo_energia": validar_real(in_consumo.value, "Consumo total", True),
                    "costo_unidad_energia": validar_real(in_costo.value, "Costo por unidad", True),
                    "limite_horas_mantenimiento": validar_real(in_limite.value, "Límite máximo de horas", False),
                    "consumo_estandar_hora": validar_real(in_estandar.value, "Consumo estándar / hora", True),
                }

                maquina = evaluar_maquina(maquina)
                flota.append(maquina)
                mostrar_informe(maquina)

                in_id.value = ""
                in_tipo.value = ""
                in_horas.value = ""
                in_consumo.value = ""
                in_costo.value = ""
                in_limite.value = ""
                in_estandar.value = ""
            except ValueError as e:
                display(HTML(f"""
                <div style="background-color: #1E293B; padding: 12px; border-radius: 8px; color: #EF4444; border-left: 4px solid #EF4444; font-family: 'Segoe UI', sans-serif;">
                    <strong>{e}</strong>
                </div>
                """))

    def on_resumen(b):
        with salida:
            clear_output()
            mostrar_resumen(flota)

    def on_comparar(b):
        with salida:
            clear_output()
            mostrar_comparacion(flota)

    def on_grafica(b):
        with salida:
            clear_output()
            mostrar_grafica(flota)

    def on_pdf(b):
        with salida:
            clear_output()
            exportar_pdf(flota)

    btn_registrar.on_click(on_registrar)
    btn_resumen.on_click(on_resumen)
    btn_comparar.on_click(on_comparar)
    btn_grafica.on_click(on_grafica)
    btn_pdf.on_click(on_pdf)

    formulario = widgets.VBox([
        encabezado,
        widgets.HTML("<div style='height: 5px;'></div>"),
        in_id,
        in_tipo,
        in_horas,
        in_consumo,
        in_costo,
        in_limite,
        in_estandar,
        widgets.HTML("<div style='height: 5px;'></div>"),
        widgets.HBox([btn_registrar, btn_resumen, btn_comparar, btn_grafica, btn_pdf], layout=widgets.Layout(margin='5px 0px 15px 0px')),
        salida
    ])

    display(formulario)


if __name__ == "__main__":
    iniciar_interfaz()

ModuleNotFoundError: No module named 'fpdf'